# Feature Definition API — Demo 1
### User Story
> *As a model developer, I want to compute a feature definition over my account population and time frame quickly so that I can see those values in a notebook.*

This notebook demonstrates:
- Defining features as **declarative dataclass objects** with a `compute()` method
- Selecting an **account population** by filtering on account-level attributes
- Parameterizing a **time window** (years for training, one day for production)
- Computing a **feature vector** across three feature types using Spark SQL

---

## 1. Setup

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("FeatureDefinitionDemo")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")

## 2. Load Raw Data

In practice these would be tables in your data catalog. Here we load from CSV files to keep the demo self-contained.

In [ ]:
accounts_df = spark.read.csv("data/accounts.csv", header=True, inferSchema=True)
transactions_df = spark.read.csv("data/transactions.csv", header=True, inferSchema=True)

# Register as Spark SQL temp views — feature compute() methods reference these by name
accounts_df.createOrReplaceTempView("accounts")
transactions_df.createOrReplaceTempView("transactions")

print(f"Accounts: {accounts_df.count():,} rows")
print(f"Transactions: {transactions_df.count():,} rows")
accounts_df.show(3)
transactions_df.show(3)

## 3. The Feature Definition API

A **feature definition** is a plain Python dataclass. It carries:

| Field | Purpose |
|---|---|
| `name` | Stable identifier shared between training and serving |
| `description` | Human-readable documentation |
| `entity` | The entity this feature is computed for (e.g. `account`) |
| `output_type` | Expected dtype of the output column |
| `compute()` | Method that accepts a `PopulationSpec` + `TimeWindow` and returns a Spark DataFrame |

The compute logic lives on the feature itself — there is no external runner to configure.

In [ ]:
from __future__ import annotations
from dataclasses import dataclass, field
from datetime import date
from typing import Literal
from pyspark.sql import DataFrame


# ---------------------------------------------------------------------------
# Core API types
# ---------------------------------------------------------------------------

@dataclass
class TimeWindow:
    """Defines the observation window for feature computation.
    
    as_of_date: the point-in-time anchor ("today" for serving, label date for training)
    lookback_days: how far back to include events (e.g. 365 = 1 year, 1 = production default)
    """
    as_of_date: date
    lookback_days: int

    @property
    def start_date(self) -> date:
        from datetime import timedelta
        return self.as_of_date - timedelta(days=self.lookback_days)

    def __repr__(self):
        return f"TimeWindow({self.start_date} → {self.as_of_date}, {self.lookback_days}d)"


@dataclass
class PopulationSpec:
    """Defines which accounts to compute features for.
    
    filters: list of SQL WHERE clause fragments applied to the accounts table
    """
    filters: list[str] = field(default_factory=list)

    def to_sql_where(self) -> str:
        if not self.filters:
            return "1=1"
        return " AND ".join(self.filters)


@dataclass
class FeatureDefinition:
    """Base class for all feature definitions."""
    name: str
    description: str
    entity: Literal["account"]
    output_type: str  # e.g. "double", "integer", "date"

    def compute(self, population: PopulationSpec, window: TimeWindow) -> DataFrame:
        raise NotImplementedError


print("API types defined: TimeWindow, PopulationSpec, FeatureDefinition")

## 4. Define Three Features

Each feature subclasses `FeatureDefinition` and implements `compute()` using Spark SQL. The SQL is readable and auditable — no black boxes.

### Feature 1 — Aggregation over time window
> *Average transaction spend over the lookback period*

In [ ]:
@dataclass
class AvgSpendFeature(FeatureDefinition):
    """Mean transaction amount over the time window."""

    def compute(self, population: PopulationSpec, window: TimeWindow) -> DataFrame:
        return spark.sql(f"""
            SELECT
                a.account_id,
                AVG(t.amount) AS {self.name}
            FROM accounts a
            LEFT JOIN transactions t
                ON  a.account_id = t.account_id
                AND t.transaction_date BETWEEN '{window.start_date}' AND '{window.as_of_date}'
            WHERE {population.to_sql_where()}
            GROUP BY a.account_id
        """)


avg_spend = AvgSpendFeature(
    name="avg_spend_in_window",
    description="Average transaction amount over the lookback window",
    entity="account",
    output_type="double",
)
print(avg_spend)

### Feature 2 — Point-in-time lookup
> *Account age in days as of the observation date*

In [ ]:
@dataclass
class AccountAgeFeature(FeatureDefinition):
    """Account age in days, computed as of window.as_of_date.
    
    This is a point-in-time lookup: it does not depend on the lookback period,
    only on the anchor date. The same definition produces correct values whether
    run for training (historical as_of_date) or serving (today).
    """

    def compute(self, population: PopulationSpec, window: TimeWindow) -> DataFrame:
        return spark.sql(f"""
            SELECT
                account_id,
                DATEDIFF(DATE('{window.as_of_date}'), opened_date) AS {self.name}
            FROM accounts
            WHERE {population.to_sql_where()}
        """)


account_age = AccountAgeFeature(
    name="account_age_days",
    description="Account age in days as of the observation date (point-in-time correct)",
    entity="account",
    output_type="integer",
)
print(account_age)

### Feature 3 — Ratio / derived feature
> *Total spend over the window as a fraction of credit limit*

In [ ]:
@dataclass
class SpendToLimitRatioFeature(FeatureDefinition):
    """Total spend in window divided by credit limit.
    
    A ratio feature that combines an aggregation (sum of spend) with
    a point-in-time account attribute (credit limit). Returns NULL
    for accounts with no transactions in the window.
    """

    def compute(self, population: PopulationSpec, window: TimeWindow) -> DataFrame:
        return spark.sql(f"""
            SELECT
                a.account_id,
                CASE
                    WHEN a.credit_limit > 0
                    THEN SUM(t.amount) / a.credit_limit
                    ELSE NULL
                END AS {self.name}
            FROM accounts a
            LEFT JOIN transactions t
                ON  a.account_id = t.account_id
                AND t.transaction_date BETWEEN '{window.start_date}' AND '{window.as_of_date}'
            WHERE {population.to_sql_where()}
            GROUP BY a.account_id, a.credit_limit
        """)


spend_to_limit = SpendToLimitRatioFeature(
    name="spend_to_limit_ratio",
    description="Sum of transaction spend in window / credit limit",
    entity="account",
    output_type="double",
)
print(spend_to_limit)

## 5. Define the Population and Time Window

The model developer specifies **who** to compute over and **when**. This is the same API call whether you're generating training data or running production scoring — only the parameters change.

In [ ]:
# Population: active credit card accounts only
population = PopulationSpec(filters=[
    "status = 'active'",
    "product_type = 'credit_card'",
])

# Time window: training mode — 2 years of history as of end of 2023
training_window = TimeWindow(
    as_of_date=date(2023, 12, 31),
    lookback_days=730,
)

# Time window: production mode — 1 day lookback as of today
production_window = TimeWindow(
    as_of_date=date.today(),
    lookback_days=1,
)

print(f"Population filter: {population.to_sql_where()}")
print(f"Training  window: {training_window}")
print(f"Production window: {production_window}")

## 6. Compute the Feature Vector

Each feature computes independently, then we join on `account_id` to produce a single feature vector DataFrame. In a production feature store this join would be managed by the platform.

In [ ]:
from functools import reduce

def compute_feature_vector(
    features: list[FeatureDefinition],
    population: PopulationSpec,
    window: TimeWindow,
) -> DataFrame:
    """Compute each feature independently and join into a single feature vector."""
    frames = [f.compute(population, window) for f in features]
    return reduce(
        lambda left, right: left.join(right, on="account_id", how="inner"),
        frames
    )


features = [avg_spend, account_age, spend_to_limit]

feature_vector = compute_feature_vector(features, population, training_window)

print(f"Feature vector schema:")
feature_vector.printSchema()

In [ ]:
feature_vector.orderBy("account_id").show(20, truncate=False)

In [ ]:
# Quick summary stats
print(f"Accounts in feature vector: {feature_vector.count():,}")
feature_vector.select(
    "avg_spend_in_window",
    "account_age_days",
    "spend_to_limit_ratio"
).summary("count", "mean", "stddev", "min", "max").show()

## 7. Switching to Production Mode

The feature definitions are **unchanged**. Only the `TimeWindow` parameter differs. This is the core reusability story: the same definition runs in training and serving.

In [ ]:
prod_vector = compute_feature_vector(features, population, production_window)

print(f"Production window: {production_window}")
print(f"Accounts scored: {prod_vector.count():,}")
prod_vector.orderBy("account_id").show(10, truncate=False)

---
## Summary

| | What the developer writes |
|---|---|
| **Feature definition** | A dataclass with `name`, `description`, `entity`, `output_type`, and `compute()` |
| **Population** | `PopulationSpec(filters=[...])` — plain SQL predicates on account columns |
| **Time window** | `TimeWindow(as_of_date=..., lookback_days=...)` — same type for training and serving |
| **Compute** | `compute_feature_vector(features, population, window)` — returns a Spark DataFrame |

The feature definition is the **single source of truth**. It ships with the model, runs in CI, and serves in production without modification.